Author<br>
**Shamim** <br>
Backend Engineer<br>
[LinkedIn](https://www.linkedin.com/posts/anamul-islam-shamim_designpattern-backend-systemdesign-activity-7415054264663814144-I_zr?utm_source=share&utm_medium=member_desktop&rcm=ACoAAD3HgNsBEHF8-0RH4iiXirsQnMi3Jq18EJQ)
<br>
Read my LinkedIn post by click on the above link about Creational Design Pattern in details

### Singleton Pattern

Sometimes you need only **ONE** object in the entire application.<br>
Examples:
* Database connection<br>
* Logger<br>
* Configuration Manager<br>
* Cache

If you allow multiple instances: <br>
* You may open multiple DB connections <br>
* Config values may differ 

**Real Life Analogy** <br>
Government ID office.<br>
Only `one central authority` issues IDs.

**Goal**<br>
Ensure a class has only one instance and provide a global access point.

#### Bad approach (no Singleton)

In [2]:
class Config:
    pass 

c1 = Config()
c2 = Config()

print(c1 is c2) # False (c1 and c2 are two different objects)

False


#### Singleton in Python (simple & clean)

In [3]:
class Singleton:
    _instance = None 
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance
    

# Usage
s1 = Singleton()
s2 = Singleton()

print(s1 is s2) # True

True


#### Real World Example
**Database Connection Pool**<br>
This is the most common example. Opening a new database connection for every single query is slow and expensive. Instead, you use a Singleton to manage a "pool" of connections.

* Why Singleton? In a real scenario, creating a connection to a database is "expensive" (it takes time and CPU). A pool creates a set of connections once and "loans" them out to different parts of your app. Using a Singleton ensures that your entire application shares the same pool, preventing you from accidentally opening hundreds of connections and crashing your database.

In [10]:
import time 


class DatabasePool:
    _instance = None 

    def __new__(cls):
        if cls._instance is None:
            print("--- Initializing Database Pool (One time only) ---")
            cls._instance = super().__new__(cls)

            # Simulate creating 3 expensive connections
            cls._instance.pool = ["Conn_A", "Conn_B", "Conn_C"]
            cls._instance.in_use = []
        return cls._instance

    def get_connection(self):
        if not self.pool:
            raise Exception("No connections available!")
        
        conn = self.pool.pop(0)
        self.in_use.append(conn)

        print(f"Loaned out: {conn}")
        return conn

    def release_connection(self, conn):
        self.in_use.remove(conn)
        self.pool.append(conn)
        print(f"Returned: {conn}")


In [11]:
# --- Application Logic ---

# Component 1 wants to query the DB
db_manager_1 = DatabasePool()
connection = db_manager_1.get_connection()

# Component 2 (somewhere else in your code)
db_manager_2 = DatabasePool()
print(f"Are they the same manager? {db_manager_1 is db_manager_2}")

connection2 = db_manager_2.get_connection()

# Clean up
db_manager_1.release_connection(connection)

--- Initializing Database Pool (One time only) ---
Loaned out: Conn_A
Are they the same manager? True
Loaned out: Conn_B
Returned: Conn_A


##### How to think as a Staff Engineer

**Use Singleton carefully:**<br>
* Global state is dangerous
* Harder to test
* Use only when multiple instances truly break the system

### Factory Pattern
You want to create Objects, but:
* You don't want to expose object creation logic<br>
* You don't want if-else everywhere<br>
* You want to easily add new types later

**Real-life Analogy**<br>
Pizza Shop<br>
You say: "Give Me Pepperoni Pizza"<br>
You don't care how it's made.

#### Bad approach (tight coupling)

In [ ]:
class EmailNotification:
    def send(self):
        print("Sending Email")
    
class SMSNotification:
    def send(self):
        print("Sending SMS")


def notify(type):
    if type == "email":
        return EmailNotification()
    elif type == "sms":
        return SMSNotification()

**Problems:**<br>
* Adding WhatsApp → modify function<br>
* Violates Open/Closed Principle

**Factory Pattern**

In [16]:
# Step-1: Common Interface
class Notification:
    def send(self):
        raise NotImplementedError

# step-2: Concrete Implementations
class EmailNotification(Notification):
    def send(self):
        print("Sending Email")


class SMSNotification(Notification):
    def send(self):
        print("Sending SMS")


# Step-3: Factory
class NotificationFactory:
    @staticmethod
    def create_notification(type):
        if type == "email":
            return EmailNotification()
        elif type == "sms":
            return SMSNotification()
        else:
            raise ValueError("Invalid notification type")
    

**Usage**

In [18]:
notification = NotificationFactory.create_notification("email")
notification.send()

Sending Email


**Staff Engineer mindset**<br>

Factory helps when:<br>
* Object creation logic is complex <br>
* You expect new types in the future <br>
* You want to reduce coupling

### Builder Pattern
Some Objects are complext to create.<br>
Examples:<br>
* User Profile
* HTTP request
* SQL query
* Computer configuration

Passing 10+ parameters into __init__ is ugly

**Real-life Analogy:**<br>
Burger Order <br>
You choose: <br>
* Bun 
* Patty
* Cheese
* Sauce <br>

Step by Step.

#### Bad approach

In [19]:
class User:
    def __init__(self, name, age, email, phone, address):
        self.name = name
        self.age = age 
        self.email = email
        self.phone = phone 
        self.address = address

**Hard to Read, Easy to Mess Up.**

#### **Builder Pattern**

In [22]:
# Step-1: Product
class User:
    def __init__(self):
        self.name = None 
        self.age = None 
        self.email = None 
        self.phone = None

# Step-2: Builder
class UserBuilder:
    def __init__(self):
        self.user = User()
    
    def set_name(self, name):
        self.user.name = name 
        return self 

    def set_age(self, age):
        self.user.age = age 
        return self
    
    def set_email(self, email):
        self.user.email = email 
        return self
    
    def build(self):
        return self.user

#### Usage (Clean & readable)

In [24]:
user = (
    UserBuilder()
    .set_name("Shamim")
    .set_age(23)
    .set_email("shamim@gmail.com")
    .build()
)

print(user.__dict__)

{'name': 'Shamim', 'age': 23, 'email': 'shamim@gmail.com', 'phone': None}


Staff Engineer mindset

**Builder is great when:**<br>
* Object has many optional fields
* You want readable code
* You want immutability later

**Quick Comparison**

| Pattern   | Solves                  | When to Use           |
| --------- | ----------------------- | --------------------- |
| Singleton | Only one instance       | DB, config, logger    |
| Factory   | Hide object creation    | Multiple object types |
| Builder   | Complex object creation | Many optional params  |
